# SmartBite YOLO26s-OBB Brazilian Expiry-Date Fine-Tuning

Fine-tunes the existing SmartBite YOLO26s-OBB expiry-date detector on the Brazilian product-expiry dataset.

Workflow:
1. Mount Drive.
2. Unzip `product-expdates-brazil-obb-split.zip`.
3. Convert selected Brazilian classes into the single production class: `expiry_date`.
4. Fine-tune from the existing SmartBite checkpoint copied from Drive.
5. Validate on val/test splits.
6. Save preview predictions and copy the full run back to Drive.

Default class mapping is conservative: `date + due -> expiry_date`. Use the config cell to include `prod` or `code` for ablation runs.


## Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


## Reset Working Directories

In [ ]:
!rm -rf /content/dataset /content/yolo_obb_dataset /content/output
!mkdir -p /content/dataset /content/yolo_obb_dataset /content/output


## Helpers

In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def run_live(cmd, env=None, cwd=None):
    cmd = [str(x) for x in cmd]
    print('>>', shlex.join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


## Config

Upload `data/product-expdates-brazil-obb-split.zip` to the same Drive folder used by the previous YOLO26s-OBB notebook:

```text
/content/drive/My Drive/sb-colab/product-expdates-brazil-obb-split.zip
```

The base checkpoint should already exist from the previous run:

```text
/content/drive/My Drive/sb-colab/yolo26s_obb_expdate_det/weights/best.pt
```


In [ ]:
DATASET_ZIP = Path('/content/drive/My Drive/sb-colab/product-expdates-brazil-obb-split.zip')
BASE_UNZIP_DIR = Path('/content/dataset')
YOLO_DATASET_ROOT = Path('/content/yolo_obb_dataset/product_expdates_brazil_expiry_obb')

# Fine-tune from the existing SmartBite expiry-date detector.
BASE_CHECKPOINT_DRIVE = Path('/content/drive/My Drive/sb-colab/yolo26s_obb_expdate_det/weights/best.pt')
YOLO_MODEL_SOURCE = str(BASE_CHECKPOINT_DRIVE)

RUNS_PROJECT = Path('/content/output')
RUN_NAME = 'smartbite_yolo26s_obb_brazil_expdate_ft_date_due'
FINAL_MODEL_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/yolo26s_obb_brazil_expdate_ft_date_due')
FINAL_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/yolo26s_obb_brazil_expdate_ft_date_due.zip')

# Brazilian source classes:
#   0: code, 1: date, 2: due, 3: prod
# Recommended first run: date + due only.
SOURCE_CLASS_IDS = {1, 2}
# Ablation options for later runs:
# SOURCE_CLASS_IDS = {1, 2, 3}      # date + due + production/manufacture date
# SOURCE_CLASS_IDS = {0, 1, 2, 3}   # high-recall all text/date-like fields

KEEP_BACKGROUND_IMAGES = True

EPOCHS = 80
IMGSZ = 1024
BATCH = 4
DEVICE = '0'  # set 'cpu' if no GPU
WORKERS = 2
PATIENCE = 20

assert DATASET_ZIP.exists(), f'Missing dataset zip: {DATASET_ZIP}'
assert BASE_CHECKPOINT_DRIVE.exists(), f'Missing base checkpoint: {BASE_CHECKPOINT_DRIVE}'
print('DATASET_ZIP =', DATASET_ZIP)
print('YOLO_MODEL_SOURCE =', YOLO_MODEL_SOURCE)
print('SOURCE_CLASS_IDS =', SOURCE_CLASS_IDS)
print('FINAL_MODEL_DRIVE_DIR =', FINAL_MODEL_DRIVE_DIR)


## Unzip Brazilian OBB Dataset

In [ ]:
run_live(['unzip', '-q', '-o', DATASET_ZIP, '-d', BASE_UNZIP_DIR])


def find_yolo_obb_dataset_root(base: Path) -> Path:
    candidates = []
    for yaml_path in base.rglob('data.yaml'):
        root = yaml_path.parent
        if (root / 'train' / 'images').exists() and (root / 'train' / 'labels').exists():
            candidates.append(root)
    assert candidates, 'Could not find YOLO OBB dataset root containing data.yaml and train/images.'
    return sorted(candidates, key=lambda p: len(str(p)))[0]


SOURCE_DATASET_ROOT = find_yolo_obb_dataset_root(BASE_UNZIP_DIR)
SOURCE_DATASET_YAML = SOURCE_DATASET_ROOT / 'data.yaml'
print('SOURCE_DATASET_ROOT =', SOURCE_DATASET_ROOT)
print(SOURCE_DATASET_YAML.read_text())

for split in ['train', 'valid', 'test']:
    image_count = len(list((SOURCE_DATASET_ROOT / split / 'images').glob('*')))
    label_count = len(list((SOURCE_DATASET_ROOT / split / 'labels').glob('*.txt')))
    print(split, 'images:', image_count, 'labels:', label_count)
    assert image_count == label_count, f'Image/label mismatch in source split: {split}'


## Convert Selected Classes To Single-Class Expiry OBB

Ultralytics OBB labels use one row per object:

```text
class_index x1 y1 x2 y2 x3 y3 x4 y4
```

This cell maps selected Brazilian source classes to class `0: expiry_date` and preserves normalized quadrilateral coordinates.


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
SOURCE_SPLITS = {
    'train': 'train',
    'val': 'valid',
    'test': 'test',
}
SOURCE_NAMES = {
    0: 'code',
    1: 'date',
    2: 'due',
    3: 'prod',
}


def parse_obb_line(line: str, label_path: Path, line_no: int) -> tuple[int, list[float]]:
    parts = line.split()
    if len(parts) != 9:
        raise ValueError(f'{label_path}:{line_no} expected 9 OBB tokens, got {len(parts)}')
    class_id = int(parts[0])
    coords = [float(value) for value in parts[1:]]
    if any(value < 0.0 or value > 1.0 for value in coords):
        raise ValueError(f'{label_path}:{line_no} has coordinate outside [0, 1]')
    return class_id, coords


def convert_label(label_path: Path) -> list[str]:
    mapped_lines = []
    raw_text = label_path.read_text(encoding='utf-8').strip()
    if not raw_text:
        return mapped_lines
    for line_no, raw_line in enumerate(raw_text.splitlines(), start=1):
        class_id, coords = parse_obb_line(raw_line, label_path, line_no)
        if class_id in SOURCE_CLASS_IDS:
            mapped_lines.append('0 ' + ' '.join(f'{value:.6f}' for value in coords))
    return mapped_lines


def convert_split(target_split: str, source_split: str) -> dict[str, int]:
    source_image_dir = SOURCE_DATASET_ROOT / source_split / 'images'
    source_label_dir = SOURCE_DATASET_ROOT / source_split / 'labels'
    image_out_dir = YOLO_DATASET_ROOT / 'images' / target_split
    label_out_dir = YOLO_DATASET_ROOT / 'labels' / target_split
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    counts = {
        'source_images': 0,
        'kept_images': 0,
        'positive_images': 0,
        'background_images': 0,
        'objects': 0,
        'dropped_no_selected_class': 0,
    }

    images = sorted(p for p in source_image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
    for idx, image_path in enumerate(images, start=1):
        counts['source_images'] += 1
        label_path = source_label_dir / f'{image_path.stem}.txt'
        assert label_path.exists(), f'Missing label: {label_path}'
        mapped_lines = convert_label(label_path)

        if not mapped_lines and not KEEP_BACKGROUND_IMAGES:
            counts['dropped_no_selected_class'] += 1
            print(f'[{target_split}] finished item {idx}/{len(images)}: dropped {image_path.name}')
            continue

        shutil.copy2(image_path, image_out_dir / image_path.name)
        out_label_path = label_out_dir / f'{image_path.stem}.txt'
        if mapped_lines:
            out_label_path.write_text('
'.join(mapped_lines) + '
', encoding='utf-8')
            counts['positive_images'] += 1
            counts['objects'] += len(mapped_lines)
        else:
            out_label_path.write_text('', encoding='utf-8')
            counts['background_images'] += 1
        counts['kept_images'] += 1
        print(f'[{target_split}] finished item {idx}/{len(images)}: kept {image_path.name} objects={len(mapped_lines)}')

    return counts


if YOLO_DATASET_ROOT.exists():
    shutil.rmtree(YOLO_DATASET_ROOT)

conversion_summary = {
    target_split: convert_split(target_split, source_split)
    for target_split, source_split in SOURCE_SPLITS.items()
}
print(json.dumps(conversion_summary, indent=2))

for split, counts in conversion_summary.items():
    assert counts['kept_images'] > 0, f'No kept images for {split}'
    assert counts['objects'] > 0, f'No positive objects for {split}; adjust SOURCE_CLASS_IDS'


## Write Ultralytics Dataset YAML

In [ ]:
DATASET_YAML = YOLO_DATASET_ROOT / 'dataset.yaml'
DATASET_YAML.write_text(
    f'''# Brazilian expiry-date OBB dataset mapped to SmartBite single-class detector
path: {YOLO_DATASET_ROOT}
train: images/train
val: images/val
test: images/test
names:
  0: expiry_date
''',
    encoding='utf-8',
)
print(DATASET_YAML.read_text())

for split in ['train', 'val', 'test']:
    images = sorted((YOLO_DATASET_ROOT / 'images' / split).glob('*'))
    labels = sorted((YOLO_DATASET_ROOT / 'labels' / split).glob('*.txt'))
    positive_labels = [p for p in labels if p.read_text(encoding='utf-8').strip()]
    print(split, 'images:', len(images), 'labels:', len(labels), 'positive labels:', len(positive_labels))
    assert images, f'No images for split: {split}'
    assert len(images) == len(labels), f'Image/label mismatch for {split}'


## Install Ultralytics

In [ ]:
run_live([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics', 'pyyaml'])
run_live(['nvidia-smi'])


## Fine-Tune YOLO26s-OBB From Existing SmartBite Checkpoint

In [ ]:
train_script = f'''
import contextlib
import os
import sys
from ultralytics import YOLO

PRINT_EVERY = 50
state = {{"batch": 0}}


def log(msg):
    print(msg, file=sys.stderr, flush=True)


def on_train_batch_end(trainer):
    state["batch"] += 1
    if state["batch"] % PRINT_EVERY == 0:
        epoch = getattr(trainer, "epoch", 0) + 1
        epochs = getattr(trainer, "epochs", "?")
        loss_items = getattr(trainer, "loss_items", None)
        log(f"epoch {{epoch}}/{{epochs}} step {{state['batch']}} loss={{loss_items}}")


def on_fit_epoch_end(trainer):
    epoch = getattr(trainer, "epoch", 0) + 1
    metrics = getattr(trainer, "metrics", None)
    log(f"epoch {{epoch}} finished metrics={{metrics}}")


model = YOLO(r"{YOLO_MODEL_SOURCE}")
model.add_callback("on_train_batch_end", on_train_batch_end)
model.add_callback("on_fit_epoch_end", on_fit_epoch_end)

with open(os.devnull, "w") as devnull, contextlib.redirect_stdout(devnull):
    results = model.train(
        data=r"{DATASET_YAML}",
        epochs={EPOCHS},
        imgsz={IMGSZ},
        batch={BATCH},
        device=r"{DEVICE}",
        project=r"{RUNS_PROJECT}",
        name=r"{RUN_NAME}",
        workers={WORKERS},
        patience={PATIENCE},
        cache=False,
        plots=True,
        close_mosaic=10,
        verbose=False,
    )

log(results)
log("Training finished.")
'''

run_live([sys.executable, '-c', train_script])


## Validate Best Checkpoint

In [ ]:
best_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'best.pt'
last_pt = RUNS_PROJECT / RUN_NAME / 'weights' / 'last.pt'
assert best_pt.exists(), f'Missing best checkpoint: {best_pt}'
print('best_pt =', best_pt)
print('last_pt =', last_pt, last_pt.exists())

val_script = f'''
from ultralytics import YOLO
model = YOLO(r"{best_pt}")
print('--- VAL ---')
metrics_val = model.val(data=r"{DATASET_YAML}", imgsz={IMGSZ}, batch={BATCH}, device=r"{DEVICE}", split='val')
print(metrics_val)
print('--- TEST ---')
metrics_test = model.val(data=r"{DATASET_YAML}", imgsz={IMGSZ}, batch={BATCH}, device=r"{DEVICE}", split='test')
print(metrics_test)
'''
run_live([sys.executable, '-c', val_script])


## Quick Prediction Preview

In [ ]:
preview_script = f'''
from pathlib import Path
from ultralytics import YOLO

model = YOLO(r"{best_pt}")
source = Path(r"{YOLO_DATASET_ROOT}") / 'images' / 'test'
results = model.predict(
    source=str(source),
    imgsz={IMGSZ},
    conf=0.05,
    device=r"{DEVICE}",
    project=r"{RUNS_PROJECT}",
    name=r"{RUN_NAME}_preview",
    save=True,
    max_det=20,
)
print('Preview images saved to:', Path(r"{RUNS_PROJECT}") / '{RUN_NAME}_preview')
print('Predicted images:', len(results))
'''
run_live([sys.executable, '-c', preview_script])


## Backup Artifacts To Drive

In [ ]:
src = RUNS_PROJECT / RUN_NAME
assert src.exists(), f'Missing run dir: {src}'
if FINAL_MODEL_DRIVE_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DRIVE_DIR)
FINAL_MODEL_DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, FINAL_MODEL_DRIVE_DIR)
print('Saved YOLO26s-OBB artifacts to:', FINAL_MODEL_DRIVE_DIR)

if FINAL_ZIP_DRIVE.exists():
    FINAL_ZIP_DRIVE.unlink()
with zipfile.ZipFile(FINAL_ZIP_DRIVE, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(FINAL_MODEL_DRIVE_DIR.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(FINAL_MODEL_DRIVE_DIR.parent))
print('Saved zip:', FINAL_ZIP_DRIVE)
print('Best checkpoint:', FINAL_MODEL_DRIVE_DIR / 'weights' / 'best.pt')


## Notes

- This notebook fine-tunes from the prior SmartBite YOLO26s-OBB expiry-date checkpoint, not from a generic pretrained model.
- Production remains single-class: `expiry_date`.
- Default mapping is `date + due`. Run ablations by changing `SOURCE_CLASS_IDS` and `RUN_NAME` together.
- Keep `test64` frozen for final local evaluation after downloading the new `weights/best.pt`.
